In [7]:
# install the required libraries
!pip install requests langchain langchain-google-genai google-generativeai

**2. Configuration: API Keys & LLM Setup**

In [3]:
# Establishing connection with google, news api and openweather api

import os
import requests
import json
from google.colab import userdata

# --- Load API Keys from Colab Secrets ---
NEWS_API_KEY = userdata.get('NEWS_API_KEY')
OPENWEATHER_API_KEY = userdata.get('OPENWEATHER_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')


# --- Configure the "Chief of Staff" LLM (Gemini) ---
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI

genai.configure(api_key=GOOGLE_API_KEY)
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash-latest",
    temperature=0.7,
    google_api_key=GOOGLE_API_KEY # <-- Explicitly pass the key here
)

print("✅ Environment is configured. Keys are loaded and LLM is ready.")

✅ Environment is configured. Keys are loaded and LLM is ready.


In [4]:
# News Retriever
def get_top_news_headlines(topic: str, api_key: str) -> list[str]:
    """
    This tool acts as our 'News Retriever Agent'.
    It fetches the top 5 news headlines for a given topic.
    """
    print(f"--- 📰 NEWS AGENT: Searching for news about '{topic}'... ---")
    url = f"https://newsapi.org/v2/everything?q={topic}&sortBy=popularity&apiKey={api_key}"

    try:
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes

        data = response.json()
        articles = data.get("articles", [])

        # Return a list of the top 5 headlines
        headlines = [article['title'] for article in articles[:5]] #you may change the number of articles according to your needs.
        print("--- 📰 NEWS AGENT: Found headlines. ---")
        return headlines

    except Exception as e:
        print(f"--- 📰 NEWS AGENT ERROR: {e} ---")
        return [f"Error fetching news: {e}"]


**4. Specialist Agent: Weather Forecaster**

In [5]:
# Weather retriever
def get_weather_forecast(city: str, api_key: str) -> str:
    """
    This tool acts as our 'Weather Forecaster Agent'.
    It gets the current weather for a specific city.
    """
    print(f"--- 🌦️ WEATHER AGENT: Getting forecast for '{city}'... ---")
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

    try:
        response = requests.get(url)
        response.raise_for_status()

        data = response.json()

        description = data['weather'][0]['description']
        temp = data['main']['temp']

        forecast = f"The current weather in {city} is {temp}°C with {description}."
        print("--- 🌦️ WEATHER AGENT: Forecast ready. ---")
        return forecast

    except Exception as e:
        print(f"--- 🌦️ WEATHER AGENT ERROR: {e} ---")
        return f"Error fetching weather: {e}"

**5. Orchestration: The Chief of Staff Agent & Daily Briefing Generation**

In [6]:
# 1. Define the parameters for our briefing
news_topic = input("What news topic are you interested in today? ")
user_city = input("Which city's weather would you like? ")

# 2. Call the specialist agents to gather data
# The "Chief of Staff" delegates the tasks
news_data = get_top_news_headlines(topic=news_topic, api_key=NEWS_API_KEY)
weather_data = get_weather_forecast(city=user_city, api_key=OPENWEATHER_API_KEY)

# Let's print the raw data to see what the specialists collected
print("\n--- RAW DATA COLLECTED ---")
print("News:", news_data)
print("Weather:", weather_data)
print("--------------------------\n")

# 3. The "Chief of Staff" (Summary Agent) synthesizes the data
# We create a detailed prompt with the raw data for the LLM
final_prompt = f"""
You are an expert Chief of Staff, preparing a personalized morning briefing for your busy executive.
Your tone should be professional yet friendly.
Combine the following raw data points into a single, cohesive, and easy-to-read summary.
Start with a pleasant greeting.

--- RAW NEWS DATA (Topic: {news_topic}) ---
{json.dumps(news_data, indent=2)}

--- RAW WEATHER DATA (for {user_city}) ---
"{weather_data}"

---

If any of the raw data contains an error message, acknowledge the error gracefully in your briefing instead of showing the raw error text. For example,
say 'I was unable to retrieve the news headlines at this time.'

Now, please write the final morning briefing for the executive.
"""

print("--- 🧠 CHIEF OF STAFF: Compiling the final briefing... ---")

final_briefing = llm.invoke(final_prompt).content

# 4. Present the final, polished result
print("\n======================================")
print("    YOUR PERSONALIZED DAILY BRIEFING    ")
print("======================================\n")
print(final_briefing)

What news topic are you interested in today? Jobs
Which city's weather would you like? Warangal
--- 📰 NEWS AGENT: Searching for news about 'Jobs'... ---
--- 📰 NEWS AGENT: Found headlines. ---
--- 🌦️ WEATHER AGENT: Getting forecast for 'Warangal'... ---
--- 🌦️ WEATHER AGENT: Forecast ready. ---

--- RAW DATA COLLECTED ---
News: ['No One Is in Charge at the US Copyright Office', 'Microsoft, OpenAI, and a US Teachers’ Union Are Hatching a Plan to ‘Bring AI into the Classroom’', 'Substack Is Having a Moment—Again. But Time Is Running Out', 'Dbrand admits it had a ‘spectacularly terrible response’ to Killswitch Joy-Con grip detachment complaints', 'Canada drops Big Tech tax to appease Trump']
Weather: The current weather in Warangal is 25.63°C with moderate rain.
--------------------------

--- 🧠 CHIEF OF STAFF: Compiling the final briefing... ---

    YOUR PERSONALIZED DAILY BRIEFING    

Good morning!

Hope you're having a productive start to your day. Here's your briefing:

**Weather:** 